# Active Learning Demo: Multimodal Conditional Distribution

This notebook demonstrates pool-based active learning for learning a **synthetic**
conditional distribution:

$$p_\star(y \mid x) = \sum_{k=1}^{K} \pi_k(x)\,\mathcal{N}\!\left(y;\,\mu_k(x),\,\Sigma_k(x)\right)$$

where all parameters ($\Omega$, $\phi$, $W$, $B$, $C$, $c$) are **fixed random functions**
of $x$ — not learned, just drawn once and held constant.

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # replace with desired GPU
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import sys
import jax
import jax.numpy as jnp
import jax.random as jr
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Make repo root and current directory importable
repo_root        = Path("../..").resolve()
multimodal_dir   = Path(".").resolve()
for p in [str(repo_root), str(multimodal_dir)]:
    if p not in sys.path:
        sys.path.insert(0, p)

%load_ext autoreload
%autoreload 2

print("JAX devices:", jax.devices())

In [ ]:
from multimodal_conditional import (
    build_distribution,
    generate_dataset,
    compute_true_nll,
    plot_output_marginals,
    plot_output_scatter_by_component,
    plot_log_prob_distribution,
    plot_mixing_weights_2d,
)

from active_learning_utils import (
    get_default_mdn_config,
    create_shared_multimodal_data,
    make_mdn_trainer_factory,
    run_pool_based_active_learning_experiment,
    plot_loss_comparison,
    plot_labeled_points_scatter,
    plot_radial_concentration,
    plot_single_run,
)

In [ ]:
# ----- Distribution hyperparameters -----
D       = 10    # input  dimension  (x ∈ R^D)
M       = 16    # output dimension  (y ∈ R^M)
L       = 4     # latent manifold dimension (L << D)
K       = 3     # true number of mixture components
P       = 128   # random Fourier features
TAU     = 1.0   # softmax temperature (used only when MIXING_MODE="random")
ALPHA   = 0.0   # variance coupling (disabled)
C_SCALE = 10.0  # mode-separation scale
DIST_SEED = 42  # fixed forever — defines the distribution

# ----- Mixing weight mode -----
# "random"     : original softmax(W @ x / tau) — uniform difficulty everywhere
# "structured" : sharp radial boundary between unimodal interior and multimodal
#                exterior — gives active learning an exploitable landscape
MIXING_MODE = "structured"

# Structured mixing hyperparameters (ignored when MIXING_MODE="random")
TRANSITION_SHARPNESS = 8.0   # beta  — higher = sharper boundary
TRANSITION_RADIUS    = 1.3   # r0    — boundary at manifold median of ||x[:L]||
ANGULAR_SHARPNESS    = 2.0   # gamma — soft angular sectors → genuine multimodality
LOGIT_SCALE          = 3.0   # scale — moderate, prevents one-hot exterior weights

# Build the fixed distribution
generate_samples, log_prob = build_distribution(
    d=D, m=M, L=L, K=K, p=P,
    tau=TAU, alpha=ALPHA, c_scale=C_SCALE,
    seed=DIST_SEED,
    mixing_mode=MIXING_MODE,
    transition_sharpness=TRANSITION_SHARPNESS,
    transition_radius=TRANSITION_RADIUS,
    angular_sharpness=ANGULAR_SHARPNESS,
    logit_scale=LOGIT_SCALE,
)

print(f"Mixing mode: {MIXING_MODE}")
print("Distribution parameters:")
params = generate_samples.get_params()
for name, arr in params.items():
    if hasattr(arr, 'shape'):
        print(f"  {name:24s}  shape={arr.shape}")
    else:
        print(f"  {name:24s}  = {arr}")

In [ ]:
key = jr.PRNGKey(0)
key, data_key = jr.split(key)

N_EXPLORE = 3_000
x_explore, y_explore = generate_dataset(
    N_EXPLORE, generate_samples,
    key=data_key, d=D, L=L, manifold_seed=1,
)

print(f"Input  shape : {x_explore.shape}   range: [{float(x_explore.min()):.3f}, {float(x_explore.max()):.3f}]")
print(f"Output shape : {y_explore.shape}   mean:  {float(y_explore.mean()):.5f}  (expected ≈ 0)")
print(f"Output std   : {float(y_explore.std()):.3f}")

---
## Step 2 — Explore the Distribution

Before running active learning it is always worth understanding the ground truth.
We look at three things:
1. **Output marginals** — do they look multimodal? Is the mean near zero?
2. **Scatter by component** — can we see distinct clusters in output space?
3. **Mixing weights in input space** — which regions are unimodal vs. multimodal?
4. **True log-likelihood** — what NLL floor should we expect from the MDN?

In [ ]:
# --- 2a. Output marginals ---
# The first four output dimensions.  Multimodal distributions produce
# histograms with more than one peak; the mean should be near 0.
plot_output_marginals(y_explore, n_dims=4, title="Output marginals (first 4 dims)")

In [ ]:
# --- 2b. Scatter coloured by dominant mixture component ---
# Each point is coloured by argmax_k π_k(x).  You should see K≈5 distinct
# clusters in the (y_0, y_1) plane when c_scale is large enough.
plot_output_scatter_by_component(
    x_explore, y_explore, generate_samples,
    K=K, tau=TAU, alpha=ALPHA,
    dim_i=0, dim_j=1,
)

In [ ]:
# # --- 2c. Mixing weights in input space ---
# plot_mixing_weights_2d(
#     x_explore, generate_samples,
#     K=K, tau=TAU, alpha=ALPHA,
#     dim_i=0, dim_j=1,
# )

In [ ]:
# 2d. True log-likelihood distribution 

plot_log_prob_distribution(log_prob, x_explore, y_explore)

true_nll_ref = compute_true_nll(log_prob, x_explore, y_explore)
print(f"True NLL (reference): {true_nll_ref:.3f}")
print("The MDN ensemble NLL should approach this value as more data is labeled.")

---
## Step 3 — Create the MDN Trainer


In [ ]:
# ----- MDN + ensemble configuration -----
ENSEMBLE_SIZE = 8
N_MIXTURES = 5     # MDN components  (> K_true = 3)
HIDDEN_FEATURES = 128
DEPTH = 2
N_ITER = 10_000   # max iterations (cap for adaptive scaling)
BATCH_SIZE = 128
ACQUISITION_BATCH = 256      # chunk size for acquisition scoring

# ----- Active learning budget -----
AL_ITERS = 20                # active learning rounds
QUERY_BATCH_SIZE = 50        # pool points labelled per round

# ----- Adaptive iteration scaling -----
ADAPTIVE_ITERS = True
ITER_PER_SAMPLE = 10

config = get_default_mdn_config(
    out_dim=M,
    hidden_features=HIDDEN_FEATURES,
    depth=DEPTH,
    num_mixtures=N_MIXTURES,
    ensemble_size=ENSEMBLE_SIZE,
)

trainer = make_mdn_trainer_factory(
    config,
    n_iter=N_ITER,
    batch_size=BATCH_SIZE,
    adaptive_iters=ADAPTIVE_ITERS,
    iter_per_sample=ITER_PER_SAMPLE,
)

print("Trainer factory created.")
print(f"  Ensemble size : {ENSEMBLE_SIZE}")
print(f"  MDN mixtures  : {N_MIXTURES}  (true K = {K})")
print(f"  Network depth : {DEPTH}  x  width {HIDDEN_FEATURES}")
print(f"  Max iters     : {N_ITER} per round")
print(f"  Adaptive iters: {ADAPTIVE_ITERS}  ({ITER_PER_SAMPLE} iter/sample)")
print(f"  AL rounds     : {AL_ITERS}  x  batch {QUERY_BATCH_SIZE}")
print(f"  Final labeled : {100 + AL_ITERS * QUERY_BATCH_SIZE}")
print()
print("Iteration schedule (first few rounds):")
for i in range(6):
    n = 100 + i * QUERY_BATCH_SIZE
    eff = min(N_ITER, ITER_PER_SAMPLE * n)
    print(f"  Round {i}: {n:>5d} labeled -> {eff:>6d} iters")

---
## Step 4 — Set Up the Shared Benchmark Dataset


In [ ]:
data = create_shared_multimodal_data(
    seed=42,
    candidate_sample_count=50_000,   # large pool so 2.2% budget is meaningful
    test_sample_count=2_000,
    initial_sample_count=100,
    d=D, m=M, L=L, K=K, p=P,
    tau=TAU, alpha=ALPHA, c_scale=C_SCALE,
    dist_seed=DIST_SEED,
    manifold_seed=1,
    mixing_mode=MIXING_MODE,
    transition_sharpness=TRANSITION_SHARPNESS,
    transition_radius=TRANSITION_RADIUS,
    angular_sharpness=ANGULAR_SHARPNESS,
    logit_scale=LOGIT_SCALE,
)

print(f"Initial labelled : {data['initial_labeled_inputs'].shape[0]}")
print(f"Unlabelled pool  : {data['remaining_pool_inputs'].shape[0]}")
print(f"Test set         : {data['test_data'][0].shape[0]}")
print(f"Input dim        : {data['initial_labeled_inputs'].shape[1]}")
print(f"Output dim       : {data['initial_labeled_targets'].shape[1]}")
print(f"True NLL (test)  : {data['true_nll']:.4f}")
print(f"Final budget     : {100 + AL_ITERS * QUERY_BATCH_SIZE} / "
      f"{data['remaining_pool_inputs'].shape[0] + 100} "
      f"({100 * (100 + AL_ITERS * QUERY_BATCH_SIZE) / (data['remaining_pool_inputs'].shape[0] + 100):.1f}%)")
print()
print("True NLL is the floor the ensemble must approach to be well-calibrated.")

---
## Step 5 — Run Active Learning


The next cell runs the full acquisition set with a shared budget, model configuration, and evaluation protocol.

Each method starts from the same initial labelled set and queries the same number of pool points per round.

In [ ]:
STRATEGIES = [
    "random",
    "mdn_epistemic_variance",
    # "sbal_mdn_epistemic_variance",  # commented out for appendix scatter run
    "mi_lb",
    # "sbal_mi_lb",  # commented out for appendix scatter run
    "bait",
    "coreset",
]

SBAL_TEMPERATURE = 0.3
CORESET_ENSEMBLE_MEMBER = 0
CORESET_POOL_SUBSAMPLE = None

results = {}
for acquisition_name in STRATEGIES:
    print(f"\n=== Running {acquisition_name} ===")
    kwargs = {}
    if acquisition_name.startswith("sbal_"):
        kwargs["sbal_temperature"] = SBAL_TEMPERATURE
    if acquisition_name == "coreset":
        kwargs.update(
            coreset_ensemble_member=CORESET_ENSEMBLE_MEMBER,
            coreset_pool_subsample=CORESET_POOL_SUBSAMPLE,
        )

    results[acquisition_name] = run_pool_based_active_learning_experiment(
        trainer,
        data,
        acquisition_name=acquisition_name,
        al_iters=AL_ITERS,
        query_batch_size=QUERY_BATCH_SIZE,
        acquisition_batch_size=ACQUISITION_BATCH,
        **kwargs,
    )

true_nll = data["true_nll"]
for name, r in results.items():
    print(f"{name:>35s}  final test NLL = {r['final_test_nll']:.4f}  "
          f"gap = {r['final_test_nll'] - true_nll:.4f}  "
          f"(train size: {r['state'].train_inputs.shape[0]})")


## Step 6 — Comparison Plots

In [ ]:
plot_loss_comparison(results)

In [ ]:
plot_labeled_points_scatter(results)

In [ ]:
# Sanity check: is the apparent left-heaviness of Random's scatter actually a
# property of the pool, or is Random biased? If Random is truly uniform, its
# acquired points should look like a low-density version of the full pool.
pool_x   = np.asarray(data["remaining_pool_inputs"])
init_x   = np.asarray(data["initial_labeled_inputs"])
full_pool = np.vstack([pool_x, init_x])

rand_state = results["random"]["state"]
rand_x     = np.asarray(rand_state.train_inputs)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))

axes[0].scatter(full_pool[:, 0], full_pool[:, 1], s=1, alpha=0.15, c="gray")
axes[0].set_title(f"Entire pool (n={len(full_pool):,})")

axes[1].scatter(rand_x[:, 0], rand_x[:, 1], s=4, alpha=0.5, c="#2171b5")
axes[1].set_title(f"Random acquired (n={len(rand_x)})")

for ax in axes:
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_aspect("equal")
    ax.set_xlabel("$x_0$"); ax.set_ylabel("$x_1$")
    ax.grid(True, alpha=0.2)

plt.suptitle("Pool density vs Random's acquired points", fontsize=13)
plt.tight_layout()
plt.show()

# Quick numeric check: mean / asymmetry of x_0 in pool vs Random.
print(f"  pool   x_0 mean = {full_pool[:, 0].mean():+.4f}  "
      f"(fraction with x_0 < 0 = {(full_pool[:, 0] < 0).mean():.3f})")
print(f"  random x_0 mean = {rand_x[:, 0].mean():+.4f}  "
      f"(fraction with x_0 < 0 = {(rand_x[:, 0] < 0).mean():.3f})")


### Query concentration near the phase boundary
The structured mixing mode places a sharp radial boundary at $\|x_{[:L]}\| = r_0 = 1.3$ separating a unimodal interior from a multimodal exterior. The bar chart below shows what fraction of each method's queries land near this boundary. The red dashed line is the fraction expected under uniform pool sampling. Bars above the line mean the method over-samples the transition zone — the information frontier.

In [ ]:
plot_radial_concentration(
    results, data,
    L=L, transition_radius=TRANSITION_RADIUS, transition_width=0.3,
);

## Inspect the MDN Model

Visualise the output distribution learned by one method.

In [ ]:
# Commented out for appendix scatter run (sbal_mi_lb not in results).
# acquisition_name = "sbal_mi_lb"
# plot_single_run(results[acquisition_name], title=acquisition_name)


In [ ]:
# Commented out for appendix scatter run (sbal_mi_lb not in results).
# # Pick a strategy to inspect
# acquisition_name_to_plot = "sbal_mi_lb"
# r = results[acquisition_name_to_plot]
# model = r["model"]
# test_inputs, test_targets = r["test_data"]

# # Sample from MDN (net 0)
# pred = model.apply(model.params, test_inputs)
# logit_weights, means, variances = pred
# mdn_samples = model.sample_from_mixture(
#     jr.PRNGKey(0), logit_weights, means, variances,
# )  # (E, N, 1, M)
# y_net0 = np.asarray(mdn_samples[0]).squeeze(1)   # (N, M)

# n_show = test_inputs.shape[0]
# true_nll_val = r["true_nll"]
# final_nll = r["final_test_nll"]
# gap = final_nll - true_nll_val

# fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# # True samples
# ax = axes[0]
# ax.scatter(np.asarray(test_targets[:n_show, 0]),
#            np.asarray(test_targets[:n_show, 1]),
#            s=5, alpha=0.4, c="steelblue")
# ax.set_xlabel("$y_0$"); ax.set_ylabel("$y_1$")
# ax.set_title("True samples  $p_\\star(y|x)$")
# ax.grid(True, alpha=0.3)

# # MDN samples (net 0)
# ax = axes[1]
# ax.scatter(y_net0[:n_show, 0], y_net0[:n_show, 1],
#            s=5, alpha=0.4, c="tomato")
# ax.set_xlabel("$y_0$"); ax.set_ylabel("$y_1$")
# ax.set_title(f"MDN samples (net 0)   NLL={final_nll:.2f}  gap={gap:.2f}")
# ax.grid(True, alpha=0.3)

# plt.suptitle(f"Predicted vs true output samples — {acquisition_name_to_plot}", fontsize=14)
# plt.tight_layout()
# plt.show()


---
## Multi-Seed Runs

For final multi-seed results, use the script runner:
```bash
python scripts/run_experiments.py --example multimodal_conditional
```